# Feature Normalization

This notebook normalizes gameplay features per player to capture relative behavior (0-1 scale).

In [1]:
import pandas as pd
import os
import numpy as np

INPUT_PATH = 'data/processed/2_cleaned_telemetry_for_modelling.csv'
OUTPUT_PATH = 'data/processed/3_normalized_telemetry.csv'

# Excluded columns (IDs, Time, Categorical)
EXCLUDE_COLS = ['_id', 'userId', 'timestamp', 'session', 'telem_unique_id', 'death_count', 'rawJson.deaths'] 
# Note: 'death_count' is a target or derived metric, might want to exclude or normalize depending on intent.
# Usually targets aren't normalized? Or is it a feature? 
# Given it's "gameplay features", and deaths is a count, I will normalize it too unless specified otherwise.
# User said "gameplay features". Death count is a feature of gameplay.

print("Loading cleaned telemetry data...")
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(df)} rows. Users: {df['userId'].nunique()}")
    
    # Identify Numeric Columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # Remove excluded
    feature_cols = [c for c in numeric_cols if c not in EXCLUDE_COLS and not c.endswith('_id')]
    print(f"Normalizing {len(feature_cols)} features: {feature_cols}")
    
    # Function to normalize group
    def normalize_group(group):
        # Check if group has variance
        # Apply MinMax: (x - min) / (max - min)
        for col in feature_cols:
            if col in group.columns:
                min_val = group[col].min()
                max_val = group[col].max()
                rng = max_val - min_val
                if rng > 0:
                    group[col] = (group[col] - min_val) / rng
                else:
                    group[col] = 0.0 # Constant value becomes 0
        return group

    # Apply Normalization per User
    print("Normalizing per user... (This may take a moment)")
    df_normalized = df.groupby('userId', group_keys=False).apply(normalize_group)

    # FIX: Restore userId if lost during groupby.apply
    if 'userId' not in df_normalized.columns:
        df_normalized['userId'] = df['userId']
    
    # Verify
    print("\n--- Verification (First User) ---")
    sample_user = df['userId'].iloc[0]
    sample_data = df_normalized[df_normalized['userId'] == sample_user]
    print(f"User: {sample_user}")
    print(sample_data[feature_cols].describe().loc[['min', 'max']])
    
    # Export
    df_normalized.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSaved normalized data to {OUTPUT_PATH}")
else:
    print(f"Error: {INPUT_PATH} not found.")

Loading cleaned telemetry data...
Loaded 2715 rows. Users: 38
Normalizing 22 features: ['itemsCollected', 'pickupAttempts', 'timeNearInteractables', 'enemiesHit', 'damageDone', 'timeInCombat', 'distanceTraveled', 'timeOutOfCombat', 'timeSprinting', 'kills', 'rawJson.items_collected', 'rawJson.pickup_attempts', 'rawJson.time_near_interactables', 'rawJson.enemies_hit', 'rawJson.damage_done', 'rawJson.time_in_combat', 'rawJson.distance_traveled', 'rawJson.time_out_of_combat', 'rawJson.time_sprinting', 'rawJson.kills', '__v_x', '__v_y']
Normalizing per user... (This may take a moment)

--- Verification (First User) ---
User: 6974892348d53c4152cf1421
     itemsCollected  pickupAttempts  timeNearInteractables  enemiesHit  \
min             0.0             0.0                    0.0         0.0   
max             1.0             1.0                    1.0         1.0   

     damageDone  timeInCombat  distanceTraveled  timeOutOfCombat  \
min         0.0           0.0               0.0        